# Bodo News Extractive Summarization System
### TF-IDF based extractive summarizer for a low-resource language (Bodo)

**Pipeline:**
1. Text cleaning & preprocessing
2. Custom Bodo sentence tokenization (split on `।`)
3. Stop-word filtering
4. TF-IDF word importance scoring
5. Sentence ranking & summary extraction (guarantees **at least 30%** of sentences are returned, so it works on any article length)
6. Reusable `summarize_article()` function for unseen articles
7. Interactive UI to paste any Bodo article and get a summary

Run the cells top to bottom.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Install/verify dependencies (most are pre-installed on Colab)
!pip install -q ipywidgets scikit-learn pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 58.8 MB/s eta 0:00:00


In [3]:
import re
import math
import pickle
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
import ipywidgets as widgets
from IPython.display import display, clear_output

pd.set_option('display.max_colwidth', 150)

##Upload the dataset



In [5]:
from google.colab import files

uploaded = files.upload()   # choose Bodo_news.csv when prompted

df = pd.read_csv('Bodo_news.csv')
print("Shape:", df.shape)
df.head()

Saving Bodo_news.csv to Bodo_news.csv
Shape: (2057, 5)


,title,article,source,url,sentence_count
0,सि ए ए जों दफा नं 6 खौ लानानै राज्यो-राजाखौ दाफोथायहो : समुज्जल भट्टाचार्य,राज्योआव दासान्दि मागु दमासि खुंफुंनायनि लाहार फाहार सोलिगासिनो दं बै दुलाराय आसाम फरायसा आफादनि दैदेनगिरि मोना सोरखारनि हेंथायै गोख्रों रावजों सा...,The Sentinel,https://bodo.sentinelassam.com/गाहाय-रादाब/dont-confuse-people-by-linking-clause-6-with-caa-samujjal-bhattacharjya,16
1,लखनौनि सा 11 राजखान्थि बादायगिरिया के बि सि आव फिन हरनायाव फेलें,बलीउदनि सावथुन मुलुगनि राफोद फावखुंगुर अमिताभ बश्शननि स्व' कौन बनेगा करोड़पति (KBC) हाबाफारिया जौसे जांख्लायाव गाखोहैदों। टि आर पि (Television Rat...,The Sentinel,https://bodo.sentinelassam.com/रंजानाय/11-politician-from-lucknow-failed-to-answer-in-kbc,21
2,बलीउदनि फावखुंगुर सालमान खाननि रेस 3 सावथुनाव सिद्धार्थ मालहत्रनि बदलै बबी देउलखौ लाबोबाय,"बलीउद सावथुन मुलुगनि राफोद फावखुंगुर सालमान खाननि रन्सायनायाव रेस 3 सावथुना गुबुन गुबुन बिथिंनिफ्राय गोबां दानख'थाइ फिन्जाथाय नुनो मोननाय जादों, न...",The Sentinel,https://bodo.sentinelassam.com/रंजानाय/bollywood-star-bobby-deol-replaced-sidharth-malhotra-in-race-3-movie-of-salman-khan,14
3,मन्थ्रि हिमन्त आ समुज्जल भट्टाचार्यखौ का & एन आर सि सायाव डिबेटाव बाहागो लानो सेलेन्ज होदों,दासान्दि बलीउदनि सावथुन मुलुगाव सोहैबाय नोगोरारि सोदांफिन्नाय आइननि दान्थे बिदान्थे गोनां आयदाया। फेस्ला फावगिरि (Comedian) अभिनिट मिस्र आ आसामनि ...,The Sentinel,https://bodo.sentinelassam.com/हरखाब-रादाब/himanta-biswa-sarma-challenges-samujjal-bhattacharjya-for-debate-over-caanrc,7
4,गुवाहाटी क्लाबाव सि ए ए नि हेंथायै आसुनि बान्जायनायाव सुबुं जाउरिखांनाय हाबाफारि,गुवाहाटी नोगोरमानि साहित्यरथी रसराज लक्ष्मीनाथ बेजबरुवा फोथाराव बृहसप्तिबार खालि रन'सिङा अदेबानि सुबुं जाउरिखां हाबाफारियाव गासैबो थाखोनि राज्यो-र...,The Sentinel,https://bodo.sentinelassam.com/गाहाय-रादाब/hundreds-participate-in-ranasinga-against-caa-at-guwahati-club,16


In [6]:
# Basic sanity check — drop empty rows, keep only needed columns
df = df.dropna(subset=['article']).reset_index(drop=True)
df['title'] = df['title'].fillna("")
print("Rows after cleaning NaNs:", len(df))

# Function to remove leading non-Devanagari content from articles and titles
def remove_leading_non_devanagari_content(text):
    if not isinstance(text, str):
        return ""
    # Remove any leading characters that are NOT Devanagari (including English letters, digits, symbols).
    # This will strip prefixes like 'GUWAHATI, May 15:' or '15 ' until the first Devanagari character.
    cleaned_text = re.sub(r'^[^ऀ-ॿ]*', '', text).strip()
    return cleaned_text

# Apply this new cleaning step to 'article' and 'title' columns
df['article'] = df['article'].apply(remove_leading_non_devanagari_content)
df['title'] = df['title'].apply(remove_leading_non_devanagari_content)

df.head()

Rows after cleaning NaNs: 2056


,title,article,source,url,sentence_count
0,सि ए ए जों दफा नं 6 खौ लानानै राज्यो-राजाखौ दाफोथायहो : समुज्जल भट्टाचार्य,राज्योआव दासान्दि मागु दमासि खुंफुंनायनि लाहार फाहार सोलिगासिनो दं बै दुलाराय आसाम फरायसा आफादनि दैदेनगिरि मोना सोरखारनि हेंथायै गोख्रों रावजों सा...,The Sentinel,https://bodo.sentinelassam.com/गाहाय-रादाब/dont-confuse-people-by-linking-clause-6-with-caa-samujjal-bhattacharjya,16
1,लखनौनि सा 11 राजखान्थि बादायगिरिया के बि सि आव फिन हरनायाव फेलें,बलीउदनि सावथुन मुलुगनि राफोद फावखुंगुर अमिताभ बश्शननि स्व' कौन बनेगा करोड़पति (KBC) हाबाफारिया जौसे जांख्लायाव गाखोहैदों। टि आर पि (Television Rat...,The Sentinel,https://bodo.sentinelassam.com/रंजानाय/11-politician-from-lucknow-failed-to-answer-in-kbc,21
2,बलीउदनि फावखुंगुर सालमान खाननि रेस 3 सावथुनाव सिद्धार्थ मालहत्रनि बदलै बबी देउलखौ लाबोबाय,"बलीउद सावथुन मुलुगनि राफोद फावखुंगुर सालमान खाननि रन्सायनायाव रेस 3 सावथुना गुबुन गुबुन बिथिंनिफ्राय गोबां दानख'थाइ फिन्जाथाय नुनो मोननाय जादों, न...",The Sentinel,https://bodo.sentinelassam.com/रंजानाय/bollywood-star-bobby-deol-replaced-sidharth-malhotra-in-race-3-movie-of-salman-khan,14
3,मन्थ्रि हिमन्त आ समुज्जल भट्टाचार्यखौ का & एन आर सि सायाव डिबेटाव बाहागो लानो सेलेन्ज होदों,दासान्दि बलीउदनि सावथुन मुलुगाव सोहैबाय नोगोरारि सोदांफिन्नाय आइननि दान्थे बिदान्थे गोनां आयदाया। फेस्ला फावगिरि (Comedian) अभिनिट मिस्र आ आसामनि ...,The Sentinel,https://bodo.sentinelassam.com/हरखाब-रादाब/himanta-biswa-sarma-challenges-samujjal-bhattacharjya-for-debate-over-caanrc,7
4,गुवाहाटी क्लाबाव सि ए ए नि हेंथायै आसुनि बान्जायनायाव सुबुं जाउरिखांनाय हाबाफारि,गुवाहाटी नोगोरमानि साहित्यरथी रसराज लक्ष्मीनाथ बेजबरुवा फोथाराव बृहसप्तिबार खालि रन'सिङा अदेबानि सुबुं जाउरिखां हाबाफारियाव गासैबो थाखोनि राज्यो-र...,The Sentinel,https://bodo.sentinelassam.com/गाहाय-रादाब/hundreds-participate-in-ranasinga-against-caa-at-guwahati-club,16


In [7]:
from google.colab import files
df.to_csv('bodo_news_cleaned.csv', index=False)
files.download('bodo_news_cleaned.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

##Text Cleaning & Preprocessing

We strip out everything except:
- Devanagari characters (Bodo is written in the Devanagari script) — Unicode range `\u0900-\u097F`
- Digits
- The Bodo sentence-ending danda `।`

This automatically removes English datelines like `GUWAHATI, May 15:`, stray symbols, and punctuation other than the danda.

In [29]:
def clean_text(text):
    """Keep only Devanagari characters, digits, danda (।), apostrophe (’ or '), and whitespace."""
    if not isinstance(text, str):
        return ""
    # Remove anything that is NOT Devanagari, a digit, danda, apostrophe (U+2019 or U+0027), or whitespace
    text = re.sub(r'[^\u0900-\u097F0-9।\s\u2019\u0027]', ' ', text)
    # Collapse multiple spaces/newlines into a single space
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Quick test
sample_raw = "GUWAHATI, May 15: बर'लोगोसे बर' आयदा गाहायनि थाखाय गोलाव सोलेगोन!! क’क्राझार ल’निडीन बर’नि बर’ बिफानाव"
print("BEFORE:", sample_raw)
print("AFTER :", clean_text(sample_raw))

BEFORE: GUWAHATI, May 15: बर'लोगोसे बर' आयदा गाहायनि थाखाय गोलाव सोलेगोन!! क’क्राझार ल’निडीन बर’नि बर’ बिफानाव
AFTER : 15 बर'लोगोसे बर' आयदा गाहायनि थाखाय गोलाव सोलेगोन क’क्राझार ल’निडीन बर’नि बर’ बिफानाव


In [31]:
# Apply cleaning to the whole dataset
df['clean_article'] = df['article'].apply(clean_text)
df['clean_title'] = df['title'].apply(clean_text)
df[['title', 'clean_article']].head()

,title,clean_article
0,सि ए ए जों दफा नं 6 खौ लानानै राज्यो-राजाखौ दाफोथायहो : समुज्जल भट्टाचार्य,राज्योआव दासान्दि मागु दमासि खुंफुंनायनि लाहार फाहार सोलिगासिनो दं बै दुलाराय आसाम फरायसा आफादनि दैदेनगिरि मोना सोरखारनि हेंथायै गोख्रों रावजों सा...
1,लखनौनि सा 11 राजखान्थि बादायगिरिया के बि सि आव फिन हरनायाव फेलें,बलीउदनि सावथुन मुलुगनि राफोद फावखुंगुर अमिताभ बश्शननि स्व' कौन बनेगा करोड़पति हाबाफारिया जौसे जांख्लायाव गाखोहैदों। टि आर पि आवबो साबसिन फिथाय नुन...
2,बलीउदनि फावखुंगुर सालमान खाननि रेस 3 सावथुनाव सिद्धार्थ मालहत्रनि बदलै बबी देउलखौ लाबोबाय,बलीउद सावथुन मुलुगनि राफोद फावखुंगुर सालमान खाननि रन्सायनायाव रेस 3 सावथुना गुबुन गुबुन बिथिंनिफ्राय गोबां दानख'थाइ फिन्जाथाय नुनो मोननाय जादों ना...
3,मन्थ्रि हिमन्त आ समुज्जल भट्टाचार्यखौ का & एन आर सि सायाव डिबेटाव बाहागो लानो सेलेन्ज होदों,दासान्दि बलीउदनि सावथुन मुलुगाव सोहैबाय नोगोरारि सोदांफिन्नाय आइननि दान्थे बिदान्थे गोनां आयदाया। फेस्ला फावगिरि अभिनिट मिस्र आ आसामनि रांखान्थि ब...
4,गुवाहाटी क्लाबाव सि ए ए नि हेंथायै आसुनि बान्जायनायाव सुबुं जाउरिखांनाय हाबाफारि,गुवाहाटी नोगोरमानि साहित्यरथी रसराज लक्ष्मीनाथ बेजबरुवा फोथाराव बृहसप्तिबार खालि रन'सिङा अदेबानि सुबुं जाउरिखां हाबाफारियाव गासैबो थाखोनि राज्यो र...


## Step 3.5: Dataset Splitting

It's crucial to split the dataset into training and testing sets **before** fitting the `TfidfVectorizer`.

This ensures that the TF-IDF statistics (word importance) are learned only from the training data. If we fit the vectorizer on the entire dataset, the test set would indirectly influence the learned word weights, leading to an overly optimistic evaluation of the summarizer's performance on unseen data. A typical split of 80% for training and 20% for testing will be used.

In [32]:
from sklearn.model_selection import train_test_split

# Split the dataset into training and testing sets
# We'll use the 'clean_article' column for TF-IDF training
# and keep the original 'article' and 'title' for summary generation and evaluation.

df_train, df_test = train_test_split(df, test_size=0.2, random_state=42)

print(f"Training set shape: {df_train.shape}")
print(f"Testing set shape: {df_test.shape}")

# Display the head of the training set
display(df_train.head())

Training set shape: (1644, 8)
Testing set shape: (412, 8)


,title,article,source,url,sentence_count,clean_article,clean_title,generated_summary
67,कक्राझार पुलिसनि जेयाव मोसौ रोगाग्रा गंबा ट्राक गारिया खाजाबाय,कक्राझार जिलानि सिङाव थानाय गुरुफेला आरो सिमुलटापु आउटपष्टनि पुलिसनि नाजानायाव गंबा ट्राक गारिया खाजाबाय। मख'जाथावदि सोलिफु सप्ताहनि बृहस्पतिबार ख...,The Sentinel,https://sentinelassam-bodo.madrid.quintype.io/गाहाय-रादाब/5-cattle-laden-trucks-seized-by-kokrajhar-police,11,कक्राझार जिलानि सिङाव थानाय गुरुफेला आरो सिमुलटापु आउटपष्टनि पुलिसनि नाजानायाव गंबा ट्राक गारिया खाजाबाय। मख'जाथावदि सोलिफु सप्ताहनि बृहस्पतिबार ख...,कक्राझार पुलिसनि जेयाव मोसौ रोगाग्रा गंबा ट्राक गारिया खाजाबाय,मख जाथावदि सोलिफु सप्ताहनि बृहस्पतिबार खालि हराव दरनै आलादा जायगायाव सिग्रोमनानै हमजाखानाय गंबा गारिनिफ्राय मा 80 मोसौ हमनो हायो।मख जाथावदि हमजाखा...
1001,दुमदुमायाव गिख्रंथाव अर खामदों,"दुमदुमा,नबेम्बर 6: थिनसुकिया जिल्लानि दुमदुमायाव थानाय टाउननि नेटाजि रोडनि महिनदार साहा बिथांनि थाखुंलिनि खाथि, कार्टाराम आगरटलानि हलसेल रेनसन गला...",The Sentinel,https://sentinelassam-bodo.madrid.quintype.io/गाहाय-रादाब/दुमदुमायाव-गिख्रंथाव-अर,4,दुमदुमा नबेम्बर 6 थिनसुकिया जिल्लानि दुमदुमायाव थानाय टाउननि नेटाजि रोडनि महिनदार साहा बिथांनि थाखुंलिनि खाथि कार्टाराम आगरटलानि हलसेल रेनसन गलानि...,दुमदुमायाव गिख्रंथाव अर खामदों,दुमदुमा नबेम्बर 6 थिनसुकिया जिल्लानि दुमदुमायाव थानाय टाउननि नेटाजि रोडनि महिनदार साहा बिथांनि थाखुंलिनि खाथि कार्टाराम आगरटलानि हलसेल रेनसन गलानि...
567,अमित साहाया बुङोदि राम मन्दिरखौ 2019 नि बिसायखथिनि सिगां लुफुंगोन: पेराला,"सुक्रबाराव, हादोरनि भारतिय जनता फार्टि हान्जानि Executive Member पेरेला सेकारजिआ बुङोदि,“आमित साहाया मिथिहोयोदि तेलेंगाना हान्जया आय़ध्यानि राम मन्...",The Sentinel,https://sentinelassam-bodo.madrid.quintype.io/गाहाय-रादाब/अमित-साहाया-बुङोदि-राम-मन,3,सुक्रबाराव हादोरनि भारतिय जनता फार्टि हान्जानि पेरेला सेकारजिआ बुङोदि आमित साहाया मिथिहोयोदि तेलेंगाना हान्जया आय़ध्यानि राम मन्दिरखौ 2019 माइथायनि...,अमित साहाया बुङोदि राम मन्दिरखौ 2019 नि बिसायखथिनि सिगां लुफुंगोन पेराला,सुक्रबाराव हादोरनि भारतिय जनता फार्टि हान्जानि पेरेला सेकारजिआ बुङोदि आमित साहाया मिथिहोयोदि तेलेंगाना हान्जया आय़ध्यानि राम मन्दिरखौ 2019 माइथायनि...
1271,मोखां बिजाबा साबसिन मेसेन्जार फोसावबाय,सान फ्रान्चिसख’/गुवाहाटी 25: सम थांनायजों लोगोसे सोशल मीडियानि गोनां आरो बाहायबोनाया बांलांनायजों लोगोसे मोजांनिफ्राय साबसिन जाबोगासिनो दं। बेबादि...,The Sentinel,https://sentinelassam-bodo.madrid.quintype.io/गाहाय-रादाब/मोखां-बिजाबा-साबसिन-मेसे,7,सान फ्रान्चिसख’ गुवाहाटी 25 सम थांनायजों लोगोसे सोशल मीडियानि गोनां आरो बाहायबोनाया बांलांनायजों लोगोसे मोजांनिफ्राय साबसिन जाबोगासिनो दं। बेबादिन...,मोखां बिजाबा साबसिन मेसेन्जार फोसावबाय,बेबादिनो मुलुगनाङैनो बाहायजानाय मुंदांखा मोखां बिजाबआ मेसेन्जारनि गेजेरजों खौरां दैथानाय आरो खौरां लालायनायफोरखौ साबसिन खालामनो थाखाय मेसेन्जार 4 ...
1665,आदिवासि आयजो सुमि टुदुखौ सिखाजों दानथारदों,"क’कराझार,नवेम्बर 14: क’कराझार जिल्लानि कछुगांव पुलिस थानानि बेन्दोंआव थानाय नाबिनगर हारिनचारा गामिनि थागिरि 60 बोसोरनि सासे आदिवासि आयजोखौ थांनाय ...",The Sentinel,https://sentinelassam-bodo.madrid.quintype.io/हरखाब-रादाब/आदिवासि-आयजो-सुमि-टुदुखौ,7,क’कराझार नवेम्बर 14 क’कराझार जिल्लानि कछुगांव पुलिस थानानि बेन्दोंआव थानाय नाबिनगर हारिनचारा गामिनि थागिरि 60 बोसोरनि सासे आदिवासि आयजोखौ थांनाय 6...,आदिवासि आयजो सुमि टुदुखौ सिखाजों दानथारदों,क कराझार नवेम्बर 14 क कराझार जिल्लानि कछुगांव पुलिस थानानि बेन्दोंआव थानाय नाबिनगर हारिनचारा गामिनि थागिरि 60 बोसोरनि सासे आदिवासि आयजोखौ थांनाय 6...


## Step 2: Custom Bodo Sentence Tokenization

Standard `nltk.sent_tokenize` is trained on English/European punctuation and won't recognise the danda (`।`). We split manually on it and drop any empty fragments.

In [33]:
def split_sentences(text):
    """Split cleaned Bodo text into a list of sentences using danda (।)."""
    if not text:
        return []
    parts = text.split('।')
    sentences = [s.strip() for s in parts if s.strip()]
    return sentences

# Quick test
test_sentences = split_sentences(df.loc[0, 'clean_article'])
print(f"Article 0 has {len(test_sentences)} sentences")
for s in test_sentences[:3]:
    print("-", s)

Article 0 has 16 sentences
- राज्योआव दासान्दि मागु दमासि खुंफुंनायनि लाहार फाहार सोलिगासिनो दं बै दुलाराय आसाम फरायसा आफादनि दैदेनगिरि मोना सोरखारनि हेंथायै गोख्रों रावजों सावरायनाय जानायखौ नुनो मोननाय जादों
- बिसोरो बुंदोंदि नोगोरारि सोदांफिन्नाय आइन जों आसाम गोरोबथा नि दफा नं 6 खौ सरजाबनो नाङा होनना हुमखि होदों
- लोगोसे गोलाव 34 बोसोर नाङै भुगिनांनाय जेंनाखौ राज्योनि थागिबि सुबुं माहारिखौ संबिजिदारि रैखाथि थामोन्थाखौ रोखा खालामनो थाखाय माखासे थांखिफोर गथायदों


## Step 3: Stop-Word Filtering

A manual list of common Bodo filler words, pronouns, and conjunctions that carry no summary-worthy meaning. We pass this straight into `TfidfVectorizer` so these words get **zero weight** during scoring.

In [34]:
BODO_STOPWORDS = [
    "आरो", "एबा", "अब्ला", "ऐ", "दा", "नो", "नों", "आं", "बिसोरो", "मोन",
    "बे", "बेफोर", "जेरै", "सोर", "बबे", "बेबादि", "आव", "जों", "नि", "सिम",
    "से", "नै", "थाम", "ब्रै", "नाथाइ", "दं", "गोन", "माब्ला", "मानोना",
    "बै", "आंनि", "थाखाय", "हागोन", "थानाय"
]
print(f"{len(BODO_STOPWORDS)} stopwords loaded")

34 stopwords loaded


## Step 4: Word Importance Scoring (TF-IDF Vectorizer)

**Important design choice:** we fit **one single `TfidfVectorizer` on the entire 2,000-article corpus**, not on each article separately. This is what makes the system generalize to *unseen* articles later — the IDF statistics (how rare/common a word is across all Bodo news) are learned once from your dataset and reused for scoring any new article you type into the UI.

`token_pattern=r'(?u)\b\w+\b'` makes sure the tokenizer correctly grabs Devanagari word characters (the default sklearn pattern requires 2+ chars and can misbehave on some scripts, so we make it explicit).

In [35]:
corpus = df_train['clean_article'].tolist()

vectorizer = TfidfVectorizer(
    stop_words=BODO_STOPWORDS,
    token_pattern=r'(?u)\b[\w\u2019\u0027]+\b', # Modified to include both Unicode (U+2019) and ASCII (U+0027) apostrophes in word characters
    lowercase=False   # Devanagari has no case, keep as-is
)

vectorizer.fit(corpus)
print("Vocabulary size:", len(vectorizer.vocabulary_))

/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['अब', 'आ', 'आर', 'इ', 'एब', 'ख', 'ग', 'ज', 'थ', 'द', 'न', 'फ', 'ब', 'बब', 'म', 'य', 'र', 'ल', 'स', 'ह'] not in stop_words.
  warnings.warn(


Vocabulary size: 3914


In [36]:
# Save the fitted vectorizer so you don't have to refit every session
with open('bodo_tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)
print("Vectorizer saved to bodo_tfidf_vectorizer.pkl")

Vectorizer saved to bodo_tfidf_vectorizer.pkl


## Step 5: Sentence Ranking & Summary Extraction

For each sentence:
1. Transform it with the **already-fitted** vectorizer (so scoring uses corpus-wide IDF, not just the single article).
2. Sum the TF-IDF weights of all its words → sentence score.
3. Rank sentences by score, descending.
4. Select the top sentences — **but instead of a fixed "top 2-3", we pick `ceil(30% of total sentences)`**, so short articles still get a meaningful summary and long articles get proportionally more sentences, always covering at least 30% of the original.
5. Re-sort the selected sentences back into their original order (so the summary reads chronologically) and join with `।`.

In [37]:
def summarize_article(raw_text, vectorizer, ratio=0.3, min_sentences=1):
    """
    Extractive summarizer for a single Bodo news article.
    Returns at least `ratio` (default 30%) of the article's sentences,
    ranked by TF-IDF importance, re-ordered chronologically.
    """
    cleaned = clean_text(raw_text)
    sentences = split_sentences(cleaned)
    n = len(sentences)

    if n == 0:
        return ""
    if n <= 2:
        # Too short to summarize meaningfully — return as-is
        return '।'.join(sentences) + '।'

    num_select = max(min_sentences, math.ceil(ratio * n))
    num_select = min(num_select, n)

    sent_vectors = vectorizer.transform(sentences)
    scores = np.asarray(sent_vectors.sum(axis=1)).flatten()

    # Indices of the top-scoring sentences
    top_idx = np.argsort(scores)[::-1][:num_select]
    # Put them back in original article order
    selected_idx = sorted(top_idx)

    summary_sentences = [sentences[i] for i in selected_idx]
    summary = '।'.join(summary_sentences) + '।'
    return summary

In [38]:
# Function to calculate compression ratio
def calculate_compression_ratio(original_text, summary_text):
    original_sentences = split_sentences(clean_text(original_text))
    summary_sentences = split_sentences(clean_text(summary_text))

    if not original_sentences:
        return 0.0 # Avoid division by zero

    num_original = len(original_sentences)
    num_summary = len(summary_sentences)

    # Compression ratio: (original - summary) / original
    # If summary is longer than original, it means negative compression or expansion
    # For consistency, return 0 if summary is longer or equal to original in sentence count
    if num_summary >= num_original:
        return 0.0
    return (num_original - num_summary) / num_original


### Test on a sample article from the dataset

In [39]:
sample_idx = 0
original = df.loc[sample_idx, 'article']
summary = summarize_article(original, vectorizer, ratio=0.3)

print("TITLE:", df.loc[sample_idx, 'title'])
print("\nORIGINAL ARTICLE:\n", original)
print("\nSUMMARY (30%):\n", summary)
print(f"\nOriginal sentences: {len(split_sentences(clean_text(original)))} | Summary sentences: {len(split_sentences(summary))}")

TITLE: सि ए ए जों दफा नं 6 खौ लानानै राज्यो-राजाखौ दाफोथायहो : समुज्जल भट्टाचार्य

ORIGINAL ARTICLE:
 राज्योआव दासान्दि मागु दमासि खुंफुंनायनि लाहार फाहार सोलिगासिनो दं बै दुलाराय आसाम फरायसा आफादनि दैदेनगिरि मोना सोरखारनि हेंथायै गोख्रों रावजों सावरायनाय जानायखौ नुनो मोननाय जादों। बिसोरो बुंदोंदि नोगोरारि सोदांफिन्नाय आइन (Citizenship Amendment Act) जों आसाम गोरोबथा (Assam Accord) नि दफा नं 6 खौ सरजाबनो नाङा होनना हुमखि होदों। लोगोसे गोलाव 34 बोसोर नाङै भुगिनांनाय जेंनाखौ राज्योनि थागिबि सुबुं माहारिखौ संबिजिदारि रैखाथि थामोन्थाखौ रोखा खालामनो थाखाय माखासे थांखिफोर गथायदों। गुवाहाटी नोगोरमानि सहीद नियासाव खौरांगिरिजों जानाय मुगा मुगि बुब्लियाव रबिबार खालि दुलाराय आसाम फरायसा आफादनि बोसोनगिरि (AASU Chief Adviser) समुज्जल भट्टाचार्य बिथाङा बुङो दफा नं 6 आ गुबैयै 1951 आरो 1971 मायथाइनि गेजेराव आसामाव हाबफैनाय आयेन बेरेखा नोगोरारिखौ सिनायथिनानै बोखारनाय आरो थागिबि सुबुं माहारिनि गांनाय उखैनायखौ सुफुंनाय आरो संबिजिदारि रैखाथि थामोनथा जगायनाय। गुबुन फारसेथिं जाय नोगोरारि सोदांफिन्नाय आइन (C

### (Optional) Generate summaries for the whole dataset

Useful if you want to eyeball quality across many articles or export results for evaluation (e.g. ROUGE against human-written summaries, if you have any).

In [40]:
# Generate summaries for the test dataset
df_test['generated_summary'] = df_test['article'].apply(lambda x: summarize_article(x, vectorizer, ratio=0.3))
df_test[['title', 'generated_summary']].head()

,title,generated_summary
184,"असम थुनलाइ आफादा गंसे हादोर, मोनसे राव खान्थिखौ सोंखारिदों",थुनलाइ आफादनि आफादगिरि पर्मानन्द राजबंची जोंनि गिबि राव एबा बिमा रावानो सिगाङाव थागोन बेनि थाखाय गुबुन रावखौ जाबसिन्नाय आयदा लानानै सावरायज्लायनान...
1724,क’कराझार जिल्लायाव सानै NDFB(S) नि सोद्रोमा हमदों,क’कराझार डिसेम्बर 10 सिरियै खौरां मोन्नानै क’कराझार जिल्ला पुलिसआ पातगांव आव थानाय 210 बेटेलियन कब्राखौ लानानै रिपु होबथानाय हाग्रामानि नं 6 आरो 7...
251,गिबि मन्थ्रि सर्बानन्द सन'वाला दैबानायाव लोमलांनाय थिलियाव गोजौ प्लेटफर्म लुफुंनो बिथोन होदों,गुवाहाटी सेप्टेमबर आसामनि गिबि मन्थ्रि आरो माजुलि बियाबनि एम एल ए सर्बानन्द सन'वाला राज्यो सोरखारनि खाजोना आरो अल'खद सामलाय बिफानखौ दैबानानि समाव ...
1121,बर’ आरो कार्बिफोरा समानै मोन्थायखौ मोन्नांगौ: प्रमद बर’,क’कराझार नवेम्बर 27 बर’ फरायसा आफादनि गाहाय आफादगिरि प्रमद बर’ बिथांआ समबाराव बुङोदि बर’ आरो कार्बिफोरा आसामनि मोनफा मोनफा थागिबि हारिसा नाथाय भार...
1486,ए एस इ बि एस सि गेलेग्रा हान्जाया सुपार लिग बिमुं आवग्रिबाय,गुवाहाटी 12 सेप्टेमबर आसाम हादोरसा मोब्लिब बर्ड गेलेमुं क्लाब नेहरु गेलेग्रा स्टुडियाम जानाय जि एस ए सुपार दिभिसन सक्कार लिग बिमुं सानराइज क्लाब ग...


### Evaluate ROUGE Scores for the Test Set

Now, we'll calculate ROUGE scores for the summaries generated for the `df_test` set. This gives us an unbiased evaluation of how well the summarizer performs on unseen articles.

In [41]:
import sys
!{sys.executable} -m pip install rouge-score
from rouge_score import rouge_scorer

def calculate_avg_rouge_scores(df_eval):
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)

    all_scores = {'rouge1': {'p': [], 'r': [], 'f': []},
                  'rouge2': {'p': [], 'r': [], 'f': []},
                  'rougeL': {'p': [], 'r': [], 'f': []}}

    for index, row in df_eval.iterrows():
        reference = row['article']
        candidate = row['generated_summary']
        scores = scorer.score(reference, candidate)

        for metric in all_scores.keys():
            all_scores[metric]['p'].append(scores[metric].precision)
            all_scores[metric]['r'].append(scores[metric].recall)
            all_scores[metric]['f'].append(scores[metric].fmeasure)

    print("Average ROUGE Scores on Test Set:")
    for metric in all_scores.keys():
        avg_precision = np.mean(all_scores[metric]['p'])
        avg_recall = np.mean(all_scores[metric]['r'])
        avg_fmeasure = np.mean(all_scores[metric]['f'])
        print(f"  {metric.upper()}:")
        print(f"    Precision: {avg_precision:.4f}")
        print(f"    Recall:    {avg_recall:.4f}")
        print(f"    F1-score:  {avg_fmeasure:.4f}")

# Calculate average ROUGE scores for the test set
calculate_avg_rouge_scores(df_test)

Average ROUGE Scores on Test Set:
  ROUGE1:
    Precision: 0.8603
    Recall:    0.3006
    F1-score:  0.4014
  ROUGE2:
    Precision: 0.4157
    Recall:    0.1729
    F1-score:  0.2224
  ROUGEL:
    Precision: 0.8603
    Recall:    0.3006
    F1-score:  0.4014


In [42]:
# Save results to CSV
df['generated_summary'] = df['article'].apply(lambda x: summarize_article(x, vectorizer, ratio=0.3))
df[['title', 'article', 'generated_summary']].to_csv('bodo_news_with_summaries.csv', index=False)
print("Saved: bodo_news_with_summaries.csv")

Saved: bodo_news_with_summaries.csv


## Step 6: Test on an UNSEEN article

Paste any Bodo news text that was **not** in your dataset to confirm the system generalizes.

In [43]:
unseen_article = """आसामनि गेलेथिलिनि बिथिङाव मोनसे मोजां खौरां। मुंदांखा राफोद लाइमोन बक्सिं गेलेगिरि सुनितपुरनि फिसाजो जमुना बर'आ बृहस्पतिबार खालि मुलुुगनां आयजो बक्सिं सेम्पियनसिप 2019 नि सेमि फाइनेलाव सोहैनो हाबाय।

रायखांनो गोनां दि कुवारटार फाइनेल खेलायाव गावनि बेरेखा बक्सिं गेलेगिरि जार्मानिनि उर्सुला गटलबखौ फेजेन्नानै 54 किलग्राम (KG) लिरथाइनि खेलायाव जमुना बर'आ सेमि फाइनेलाव गावनि मासि आवग्रिनो हायो।

कुवारटार फाइनेल खेलायाव जार्मानिनि बक्सार गोख्रों गेलेगिरिखौ 4-1 पइन्टाव फेजेन्नानै देरहानाय नि माला गाननो हासिनो 22 बोसोर बैसोनि आसामनि सुनितपुरनि फिसाजो जमुना बर'आ।

फैगो 12 अक्ट'बराव रासियायाव खुंफुं जानो गोनां सेमि फाइनेल खेलायाव गेलेगिरि जमुना बर'आ साइनिस टाइपेनि हेंथायै गेलेमु दावहायाव मुगा मुगि जुजिगोन।

रायखांनो गोनां दि बे सेम्पियनसिप खेलायाव गिबिसिननि थाखाय सेमि फाइनेलाव सोहैनाया भारतनि थाखाय मोनथाम बान्था दखल खालामनायखौ रोखा खालाम फिनबाय।

मनिपुरनि फिसाजो मेरि कम आरो मन्जु रानिनि जोहै बादायलायनायाव दासिमाव भारत हादोरा मोननै बान्था आवग्रिनायखौ रोखा खालामदों। राज्योनि गुबुन सासे लाइमोन आयजो बक्सिं गेलेगिरि गलाधाटनि लाभलीना बरगहाइयाबो गावनि बेरेखा गेलेगिरि फेजेन्नानै कुवार टार फाइनेलाव सोहैनो हादों। सिगांनि खेलाफोराव जमुना बर'आ मंगलियान नि गोख्रों बक्सिं गेलेगिरि मिसिद्मा आर्नेदलाइ आरो आलजिरियानि गोख्रों बक्सिं गेलेगिरि फाउथ आबिदादखौ फेजेन्नानै देरहानायनि माला गान्नानै रोखा खालामदोंमोन।"""  # <-- replace with a real unseen Bodo article

print(summarize_article(unseen_article, vectorizer, ratio=0.3))

रायखांनो गोनां दि कुवारटार फाइनेल खेलायाव गावनि बेरेखा बक्सिं गेलेगिरि जार्मानिनि उर्सुला गटलबखौ फेजेन्नानै 54 किलग्राम लिरथाइनि खेलायाव जमुना बर'आ सेमि फाइनेलाव गावनि मासि आवग्रिनो हायो।कुवारटार फाइनेल खेलायाव जार्मानिनि बक्सार गोख्रों गेलेगिरिखौ 4 1 पइन्टाव फेजेन्नानै देरहानाय नि माला गाननो हासिनो 22 बोसोर बैसोनि आसामनि सुनितपुरनि फिसाजो जमुना बर'आ।मनिपुरनि फिसाजो मेरि कम आरो मन्जु रानिनि जोहै बादायलायनायाव दासिमाव भारत हादोरा मोननै बान्था आवग्रिनायखौ रोखा खालामदों।


In [44]:
# Calculate compression ratio for the unseen article
compression_ratio_unseen = calculate_compression_ratio(unseen_article, summarize_article(unseen_article, vectorizer, ratio=0.3))
print(f"Compression ratio for the unseen article: {compression_ratio_unseen:.4f}")

Compression ratio for the unseen article: 0.6667


## Step 7: Interactive UI

Paste any Bodo article into the box, adjust the summary-length slider if you want more/less than 30%, and click **Summarize**.

In [45]:
article_input = widgets.Textarea(
    value='',
    placeholder='Paste a Bodo news article here...',
    layout=widgets.Layout(width='100%', height='220px')
)

ratio_slider = widgets.FloatSlider(
    value=0.3, min=0.1, max=0.9, step=0.05,
    description='Summary %:',
    style={'description_width': 'initial'}
)

summarize_button = widgets.Button(
    description='Summarize',
    button_style='success',
    icon='compress'
)

output_box = widgets.Output()

def on_summarize_clicked(b):
    with output_box:
        clear_output()
        text = article_input.value.strip()
        if not text:
            print("⚠️  Please paste an article first.")
            return
        result = summarize_article(text, vectorizer, ratio=ratio_slider.value)
        n_orig = len(split_sentences(clean_text(text)))
        n_summ = len(split_sentences(result))
        print(f"Original sentences: {n_orig}  |  Summary sentences: {n_summ}  ({ratio_slider.value:.0%} target)\n")
        print("SUMMARY:\n")
        print(result)

summarize_button.on_click(on_summarize_clicked)

display(widgets.VBox([
    widgets.HTML("<h3>Bodo News Extractive Summarizer</h3>"),
    article_input,
    ratio_slider,
    summarize_button,
    output_box
]))

In [46]:
from google.colab import files
files.download('bodo_tfidf_vectorizer.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>